In [25]:
import polars as pl
import pandas as pd
import numpy as mp
import altair as alt
import pyarrow
from func.UsefulFunctions import prettifyGenres

This notebook takes the group data (that was extracted from the Music Brainz JSON) and inspects it. Groups are groups, choirs, orchestras unless stated otherwise.

In [26]:
groupDF = pl.read_parquet("/home/aja25/Documents/fromCluster/ensemblesMusicBrainzData.parquet")

#drop the relations table as it is no longer needed
groupDF = groupDF.drop("relations")
#De-duplicate the data
groupDF = groupDF.unique()

num_of_ids = groupDF.select(
    pl.col("id")
).unique().count().item(0,0)
print(f"There are {num_of_ids} unique IDs in the dataset (that is a snapshot from 18/07/2026)")


There are 672717 unique IDs in the dataset (that is a snapshot from 18/07/2026)


In [27]:
#How many people do not have a name in this dataset?
num_people_with_no_names = groupDF.select(
    pl.col("name"),
    pl.col("id")
).unique().null_count()
print(num_people_with_no_names)
print(f"There are {num_people_with_no_names.item(0,0)} entities with no name, and {num_people_with_no_names.item(0,1)} entities with no ids.")

shape: (1, 2)
┌──────┬─────┐
│ name ┆ id  │
│ ---  ┆ --- │
│ u32  ┆ u32 │
╞══════╪═════╡
│ 0    ┆ 0   │
└──────┴─────┘
There are 0 entities with no name, and 0 entities with no ids.


In [28]:
#check if there are any duplicates in ids
ids = groupDF.select(
    pl.col("id")
).is_duplicated()
print(ids.value_counts())

print("Based off of the lack of trues in the is_duplicated function. All rows are unique after the deduplication ")

shape: (1, 2)
┌───────┬────────┐
│       ┆ count  │
│ ---   ┆ ---    │
│ bool  ┆ u32    │
╞═══════╪════════╡
│ false ┆ 672717 │
└───────┴────────┘
Based off of the lack of trues in the is_duplicated function. All rows are unique after the deduplication 


In [29]:
#Do a count for the countries for the dataset.
countryCount = groupDF.select(pl.col("country").value_counts(sort=True)).unnest("country")
print(countryCount)

shape: (236, 2)
┌─────────┬────────┐
│ country ┆ count  │
│ ---     ┆ ---    │
│ str     ┆ u32    │
╞═════════╪════════╡
│ null    ┆ 203557 │
│ US      ┆ 121124 │
│ GB      ┆ 45511  │
│ DE      ┆ 33522  │
│ JP      ┆ 29454  │
│ …       ┆ …      │
│ AN      ┆ 1      │
│ BQ      ┆ 1      │
│ GQ      ┆ 1      │
│ SS      ┆ 1      │
│ WF      ┆ 1      │
└─────────┴────────┘


Based off the table, the most common countries are (outside of null which shows that most groups do not have a country attached to them is) US, GB, DE, and japan

In [30]:
#What is the most common type of group for the entities
typeCount = groupDF.group_by("type").agg(pl.col("id").count())
typeCountPercent = groupDF.select(pl.col("type").value_counts(sort=True, normalize=True, name="Fraction")).unnest("type")
print(typeCount)

typeChart = (
    alt.Chart(typeCount).mark_bar().encode(
        y="id",
        x="type",
    )
    .properties(width=500, title="Frequency of Types on Music Brainz")
)
typeChart.encoding.x.title = "Type"
typeChart.encoding.y.title = "Count"
typeChart.show()

shape: (3, 2)
┌───────────┬────────┐
│ type      ┆ id     │
│ ---       ┆ ---    │
│ str       ┆ u32    │
╞═══════════╪════════╡
│ Group     ┆ 650186 │
│ Choir     ┆ 10771  │
│ Orchestra ┆ 11760  │
└───────────┴────────┘


alt.Chart(...)

In [31]:
print(typeCountPercent)

shape: (3, 2)
┌───────────┬──────────┐
│ type      ┆ Fraction │
│ ---       ┆ ---      │
│ str       ┆ f64      │
╞═══════════╪══════════╡
│ Group     ┆ 0.966507 │
│ Orchestra ┆ 0.017481 │
│ Choir     ┆ 0.016011 │
└───────────┴──────────┘


The biggest type in this group is "Group" by a large percent (96.7%), and the other ensemble types exist around 1.7% and 1.6%.

In [33]:
#Let's get the genres out

groupDFPandas = groupDF.to_pandas(use_pyarrow_extension_array=True)
groupDFPandas['genreList'] = groupDFPandas['genres'].map(prettifyGenres)
groupDF = pl.from_pandas(groupDFPandas)
display(groupDF.head(3))

id,name,type,gender,country,genres,isnis,ipis,life-span,genreList
str,str,str,str,str,list[struct[4]],list[str],list[str],struct[3],list[str]
"""db4a8aac-7dd6-4053-961f-c0b486…","""Alley""","""Group""",null,"""RU""","[{""eacfa027-2fad-413f-a2f1-80fa43674f0b"",1,"""",""death metal""}]",[],[],"{null,""2005-12"",false}","[""death metal""]"
"""d500be31-37ab-4c87-bad5-e0c80e…","""Hedtrip""","""Group""",null,null,[],[],[],"{null,null,false}",[]
"""49c2a65f-3746-46e1-8963-23a7dd…","""Sisterhood""","""Group""",null,null,[],[],[],"{null,null,false}",[]


In [35]:
#Get the most common genre
genreCount = groupDF.select(
    pl.col("id"),
    pl.col("name"),
    pl.col("type"),
    pl.col("genreList").alias("Genre"),
)

genreCount = genreCount.explode("Genre", empty_as_null=True, keep_nulls=True)
genreCount = genreCount.unique()
number_of_genres = genreCount.select(
    pl.col("Genre").value_counts(sort=True)
).unnest("Genre")
print(number_of_genres)


shape: (1_339, 2)
┌──────────────────────┬────────┐
│ Genre                ┆ count  │
│ ---                  ┆ ---    │
│ str                  ┆ u32    │
╞══════════════════════╪════════╡
│ null                 ┆ 571747 │
│ rock                 ┆ 11429  │
│ punk                 ┆ 9985   │
│ black metal          ┆ 6911   │
│ metal                ┆ 6832   │
│ …                    ┆ …      │
│ alpenrock            ┆ 1      │
│ expressionism        ┆ 1      │
│ andalusian classical ┆ 1      │
│ kawaii future bass   ┆ 1      │
│ salsa dura           ┆ 1      │
└──────────────────────┴────────┘


In [36]:
#Look for the type and genre split using group by.
genreCount.group_by(["type",'Genre']).agg(pl.col('id').count())

type,Genre,id
str,str,u32
"""Group""","""wonky techno""",2
"""Group""","""hardcore hip hop""",61
"""Group""","""hardstyle""",91
"""Group""","""forró universitário""",10
"""Group""","""vapornoise""",2
…,…,…
"""Group""","""latin disco""",2
"""Group""","""rap metal""",82
"""Group""","""cumbia villera""",7


In [37]:
#Get the group and country count
groupDF.group_by(["type","country"]).agg(pl.col('id').count())

type,country,id
str,str,u32
"""Orchestra""","""GE""",9
"""Group""","""BT""",3
"""Group""","""IN""",719
"""Orchestra""","""ID""",69
"""Group""","""AS""",3
…,…,…
"""Choir""","""NG""",3
"""Group""","""ET""",33
"""Choir""","""HK""",32
